# Photos Library Duplicate Cleanup Notebook

Report-only v1. This notebook does **not** delete anything.

Run order:
1. Run configuration.
2. Load/build inventory.
3. Fill identity fields.
4. Group and analyze duplicate candidates.
5. Write permanent operation report.

Helper functions live in `photos_duplicate_cleanup_helpers.py` so the notebook stays readable.


In [ ]:
# ============================================================
# Cell 1. Configuration
# ============================================================

from pathlib import Path
import os
import sys
import json
from datetime import datetime

PROJECT_ROOT = Path("/Users/huohsien/workspace/python/explore_photos_library")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

LIBRARY_ID = "current_default"
LIBRARY_PATH = Path("/Users/huohsien/Pictures/Photos Library.photoslibrary")

CACHE_DIR = PROJECT_ROOT / "data" / "inventory_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_NAME = LIBRARY_ID
INVENTORY_CACHE_PATH = CACHE_DIR / f"{CACHE_NAME}.inventory.pkl.gz"

REPORTS_ROOT = PROJECT_ROOT / "IMPORTANT_Photos_Library_Critical_Operation_Reports"
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d-%H%M%S")
REPORT_DIR = REPORTS_ROOT / f"{RUN_TIMESTAMP}__Photos_Library_Duplicate_Cleanup__{LIBRARY_ID}"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("LIBRARY_ID:", LIBRARY_ID)
print("LIBRARY_PATH:", LIBRARY_PATH)
print("INVENTORY_CACHE_PATH:", INVENTORY_CACHE_PATH)
print("REPORT_DIR:", REPORT_DIR)


In [ ]:
# ============================================================
# Cell 2. Load or build inventory
# ============================================================

import osxphotos
from photos_inventory import (
    build_inventory,
    print_inventory_summary,
    save_inventory_cache,
    load_inventory_cache,
)

FORCE_REBUILD_INVENTORY = True

if INVENTORY_CACHE_PATH.exists() and not FORCE_REBUILD_INVENTORY:
    inventory = load_inventory_cache(
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )
    print("Loaded inventory cache:", INVENTORY_CACHE_PATH)
else:
    print("Building inventory from Photos Library:")
    print(LIBRARY_PATH)

    photosdb = osxphotos.PhotosDB(dbfile=str(LIBRARY_PATH))
    osx_assets = photosdb.photos()

    inventory = build_inventory(osx_assets)

    save_inventory_cache(
        inventory=inventory,
        cache_name=CACHE_NAME,
        cache_dir=CACHE_DIR,
    )

    print("Saved inventory cache:", INVENTORY_CACHE_PATH)

print_inventory_summary(inventory)


In [ ]:
# ============================================================
# Cell 3. Fill identity fields
# ============================================================

from photos_duplicate_cleanup_helpers import fill_duplicate_cleanup_identity_fields

fill_duplicate_cleanup_identity_fields(inventory)


In [ ]:
# ============================================================
# Cell 4. Group duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import group_assets_by_field

unique_id_groups, assets_without_unique_id = group_assets_by_field(
    inventory,
    "photo_library_asset_unique_id",
)

duplicate_candidate_groups = {
    unique_id: group
    for unique_id, group in unique_id_groups.items()
    if len(group) > 1
}

print("assets:", len(inventory["assets"]))
print("generated unique_id count:", len(unique_id_groups))
print("assets without unique_id:", len(assets_without_unique_id))
print("duplicate candidate group count:", len(duplicate_candidate_groups))
print("duplicate candidate asset count:", sum(len(group) for group in duplicate_candidate_groups.values()))

if assets_without_unique_id:
    print()
    print("First assets without unique_id:")
    for asset in assets_without_unique_id[:10]:
        print(
            asset.get("original_filename"),
            asset.get("uuid"),
            asset.get("asset_scope"),
            asset.get("path"),
        )


In [ ]:
# ============================================================
# Cell 5. Analyze duplicate candidates
# ============================================================

from photos_duplicate_cleanup_helpers import analyze_duplicate_candidate_groups

duplicate_analysis = analyze_duplicate_candidate_groups(duplicate_candidate_groups)


In [ ]:
# ============================================================
# Cell 6. Summary
# ============================================================

from photos_duplicate_cleanup_helpers import count_records_by_status

status_counts = count_records_by_status(duplicate_analysis)

delete_candidate_count = sum(
    len(record.get("delete_candidates") or [])
    for record in duplicate_analysis
)

print("status counts:")
for status, count in status_counts.items():
    print(f"  {status}: {count}")

print("delete candidate asset count:", delete_candidate_count)

print()
print("First deletable duplicate groups:")
printed = 0

for record in duplicate_analysis:
    if record.get("status") != "DELETABLE_DUPLICATE":
        continue

    print("-" * 80)
    print("reason:", record.get("reason"))
    print("asset_count:", record.get("asset_count"))
    print("keep_assets:", len(record.get("keep_assets") or []))
    print("delete_candidates:", len(record.get("delete_candidates") or []))

    for asset in (record.get("keep_assets") or []):
        print("  KEEP:", asset["original_filename"], asset["date_added"], asset["path"])

    for asset in (record.get("delete_candidates") or []):
        print("  DELETE:", asset["original_filename"], asset["date_added"], asset["path"])

    printed += 1

    if printed >= 10:
        print("... more groups not printed")
        break


In [ ]:
import inspect
import importlib
import photos_duplicate_cleanup_helpers as cleanup_helpers

cleanup_helpers = importlib.reload(cleanup_helpers)

source = inspect.getsource(cleanup_helpers.write_operation_report)

print("helper file:", cleanup_helpers.__file__)
print("safety_counts:", "safety_counts" in source)
print("duplicate_review_assets.tsv:", "duplicate_review_assets.tsv" in source)
print("assets_without_unique_id.tsv:", "assets_without_unique_id.tsv" in source)

In [ ]:
# ============================================================
# Cell 7. Write permanent operation report
# ============================================================

report_result = cleanup_helpers.write_operation_report(
    report_dir=REPORT_DIR,
    duplicate_analysis=duplicate_analysis,
    inventory=inventory,
    assets_without_unique_id=assets_without_unique_id,
    duplicate_candidate_groups=duplicate_candidate_groups,
    run_timestamp=RUN_TIMESTAMP,
    library_id=LIBRARY_ID,
    library_path=LIBRARY_PATH,
    inventory_cache_path=INVENTORY_CACHE_PATH,
)

delete_candidate_rows = report_result["delete_candidate_rows"]
keep_asset_rows = report_result["keep_asset_rows"]
duplicate_review_asset_rows = report_result["duplicate_review_asset_rows"]
location_conflict_rows = report_result["location_conflict_rows"]
live_photo_candidate_rows = report_result["live_photo_candidate_rows"]
assets_without_unique_id_rows = report_result["assets_without_unique_id_rows"]
status_counts = report_result["status_counts"]
safety_counts = report_result["safety_counts"]

In [ ]:
# ============================================================
# Cell 8. Optional: print manual deletion list
# ============================================================
#
# This notebook does NOT delete anything from Photos Library.
# It only produces a report and delete candidate list.
#
# For actual deletion, review delete_candidates.tsv first.

for row in delete_candidate_rows[:100]:
    print(
        row["original_filename"],
        row["date"],
        row["date_added"],
        row["path"],
    )

if len(delete_candidate_rows) > 100:
    print("... more delete candidates not printed")
